In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn

In [6]:
df=pd.read_csv('/content/product_metadata (3).csv')

In [3]:
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   product_id       150 non-null    object
 1   category         146 non-null    object
 2   claims           150 non-null    object
 3   ingredient_tags  139 non-null    object
 4   pack_size        150 non-null    object
dtypes: object(5)
memory usage: 6.0+ KB


In [8]:
df.duplicated().sum()

np.int64(0)

In [10]:
df['product_id'].duplicated().sum()

np.int64(0)

In [9]:
df.sample(10)

,product_id,category,claims,ingredient_tags,pack_size
120,P121,Proten Shake,"kosher, jain-friendly, gluten-free, high-protein",whey,4-Pack
141,P142,Protein Shake,"gluten-free, high-protein, jain-friendly, kosher","almond-flour, oats",12-Pack
15,P016,Protein Shake,nut-free,soy,Single
143,P144,Meal Replacement,"halal, diabetic-friendly, low-sugar","monk-fruit, collagen",4-Pack
68,P069,Meal Replacement,plant-based,oats,Single
140,P141,Healthy Snack,"gluten-free, kosher, diabetic-friendly","almond-flour, collagen",4-Pack
14,P015,Meal Replacement,"clean-label, diabetic-friendly","soy, stevia, monk-fruit",Single
113,P114,Healthy Snack,"nut-free, jain-friendly, clean-label, keto-fri...","monk-fruit, oats",Single
35,P036,Protein Bar,high-protein,"oats, whey, pea-protein",4-Pack
112,P113,Meal Replacement,vegan,whey,4-Pack


In [11]:
df['category'].value_counts()

,count
category,
Meal Replacement,24
Healthy Snack,20
Proten Shake,19
Protein Bar,19
Electrolyte Drink,18
Protein bar,17
Supplement,16
Protein Shake,13


In [12]:
df['category'] = df['category'].replace({
    'Proten Shake': 'Protein Shake',
    'Protein bar': 'Protein Bar'
})

In [13]:
df['category'].value_counts()

,count
category,
Protein Shake,32
Meal Replacement,24
Healthy Snack,20
Protein Bar,19
Electrolyte Drink,18
Protein bar,17
Supplement,16


In [14]:
# Remove leading/trailing whitespace
df['category'] = df['category'].str.strip()
df['category'] = df['category'].replace({
    'Proten Shake': 'Protein Shake',
    'Protein bar': 'Protein Bar'
})
print(df['category'].value_counts())

category
Protein Bar          36
Protein Shake        32
Meal Replacement     24
Healthy Snack        20
Electrolyte Drink    18
Supplement           16
Name: count, dtype: int64


In [15]:
df['pack_size'].value_counts()

,count
pack_size,
Single,96
4-Pack,36
12-Pack,18


In [16]:
df['ingredient_tags'].value_counts()

,count
ingredient_tags,
monk-fruit,8
whey,8
pea-protein,7
soy,6
oats,5
collagen,4
almond-flour,4
stevia,4
"soy, almond-flour",4


In [17]:
# Get all unique ingredient names
unique_ingredients = set()
for tags in df['ingredient_tags'].dropna():
    ingredients = [x.strip() for x in tags.split(',')]
    unique_ingredients.update(ingredients)
unique_ingredients = sorted(unique_ingredients)
print(unique_ingredients)

['almond-flour', 'collagen', 'monk-fruit', 'oats', 'pea-protein', 'soy', 'stevia', 'whey']


In [18]:
#Create column like OHE
df['ingredient_tags'] = (
    df['ingredient_tags']
    .str.replace(r'\s*,\s*',',',regex=True)
    .str.strip()
)
ingredient_ohe = df['ingredient_tags'].str.get_dummies(sep=',')
df = pd.concat([df, ingredient_ohe], axis=1)

In [19]:
df.sample(5)

,product_id,category,claims,ingredient_tags,pack_size,almond-flour,collagen,monk-fruit,oats,pea-protein,soy,stevia,whey
50,P051,Protein Shake,"vegan, gluten-free, halal","collagen,soy",4-Pack,0,1,0,0,0,1,0,0
97,P098,Supplement,"kosher, nut-free",soy,Single,0,0,0,0,0,1,0,0
25,P026,Electrolyte Drink,"gluten-free, diabetic-friendly, vegan","almond-flour,whey",4-Pack,1,0,0,0,0,0,0,1
21,P022,Protein Bar,"gluten-free, vegan, clean-label, keto-friendly","oats,whey",Single,0,0,0,1,0,0,0,1
36,P037,Supplement,"halal, keto-friendly, low-sugar","soy,almond-flour",4-Pack,1,0,0,0,0,1,0,0


In [20]:
df.drop(columns=['ingredient_tags'],inplace=True)

In [21]:
df.sample(5)

,product_id,category,claims,pack_size,almond-flour,collagen,monk-fruit,oats,pea-protein,soy,stevia,whey
48,P049,Protein Shake,"high-protein, gluten-free, diabetic-friendly, ...",Single,0,0,0,0,1,0,0,0
85,P086,Protein Bar,"plant-based, gluten-free, nut-free",Single,0,1,1,0,0,1,0,0
3,P004,Healthy Snack,vegan,4-Pack,0,0,0,0,0,0,1,0
79,P080,Healthy Snack,"jain-friendly, clean-label, keto-friendly",Single,0,0,0,0,0,0,0,1
115,P116,Protein Shake,"plant-based, gluten-free",Single,0,1,0,1,0,0,0,1


In [22]:
# Get all unique iclaims
unique_ingredients = set()
for tags in df['claims'].dropna():
    ingredients = [x.strip() for x in tags.split(',')]
    unique_ingredients.update(ingredients)
unique_ingredients = sorted(unique_ingredients)
print(unique_ingredients)

['clean-label', 'diabetic-friendly', 'gluten-free', 'halal', 'high-protein', 'jain-friendly', 'keto-friendly', 'kosher', 'low-sugar', 'nut-free', 'plant-based', 'vegan']


In [23]:
df['claims'] = (
    df['claims']
    .str.replace(r'\s*,\s*',',',regex=True)
    .str.strip()
)
ingredient_ohe = df['claims'].str.get_dummies(sep=',')
df = pd.concat([df, ingredient_ohe], axis=1)

In [24]:
df.sample(5)

,product_id,category,claims,pack_size,almond-flour,collagen,monk-fruit,oats,pea-protein,soy,stevia,whey,clean-label,diabetic-friendly,gluten-free,halal,high-protein,jain-friendly,keto-friendly,kosher,low-sugar,nut-free,plant-based,vegan
14,P015,Meal Replacement,"clean-label,diabetic-friendly",Single,0,0,1,0,0,1,1,0,1,1,0,0,0,0,0,0,0,0,0,0
27,P028,Protein Shake,"kosher,plant-based,clean-label,gluten-free",Single,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,1,0
133,P134,Protein Bar,"clean-label,high-protein,nut-free",Single,0,1,1,0,0,0,0,1,1,0,0,0,1,0,0,0,0,1,0,0
39,P040,Supplement,"high-protein,jain-friendly,nut-free",Single,1,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,1,0,0
119,P120,Protein Bar,"clean-label,high-protein,jain-friendly",Single,0,0,0,0,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0


In [25]:
df.drop(columns=['claims'],inplace=True)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   product_id         150 non-null    object
 1   category           146 non-null    object
 2   pack_size          150 non-null    object
 3   almond-flour       150 non-null    int64 
 4   collagen           150 non-null    int64 
 5   monk-fruit         150 non-null    int64 
 6   oats               150 non-null    int64 
 7   pea-protein        150 non-null    int64 
 8   soy                150 non-null    int64 
 9   stevia             150 non-null    int64 
 10  whey               150 non-null    int64 
 11  clean-label        150 non-null    int64 
 12  diabetic-friendly  150 non-null    int64 
 13  gluten-free        150 non-null    int64 
 14  halal              150 non-null    int64 
 15  high-protein       150 non-null    int64 
 16  jain-friendly      150 non-null    int64 
 1

In [27]:
df[df['category'].isnull()]

,product_id,category,pack_size,almond-flour,collagen,monk-fruit,oats,pea-protein,soy,stevia,whey,clean-label,diabetic-friendly,gluten-free,halal,high-protein,jain-friendly,keto-friendly,kosher,low-sugar,nut-free,plant-based,vegan
54,P055,NaN,12-Pack,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0
64,P065,NaN,Single,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,1,0,0
71,P072,NaN,4-Pack,0,0,1,0,0,0,1,0,0,0,1,1,0,0,0,1,1,0,0,0
77,P078,NaN,Single,0,0,0,0,0,0,0,1,1,0,0,1,0,0,1,0,0,0,0,1


In [28]:
categories = df['category'].dropna().unique()
df['category'] = df['category'].apply(
    lambda x: np.random.choice(categories) if pd.isna(x) else x
)

In [29]:
df[df['category'].isnull()]

,product_id,category,pack_size,almond-flour,collagen,monk-fruit,oats,pea-protein,soy,stevia,whey,clean-label,diabetic-friendly,gluten-free,halal,high-protein,jain-friendly,keto-friendly,kosher,low-sugar,nut-free,plant-based,vegan


In [30]:
df.sample(5)

,product_id,category,pack_size,almond-flour,collagen,monk-fruit,oats,pea-protein,soy,stevia,whey,clean-label,diabetic-friendly,gluten-free,halal,high-protein,jain-friendly,keto-friendly,kosher,low-sugar,nut-free,plant-based,vegan
95,P096,Electrolyte Drink,4-Pack,0,1,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0
100,P101,Protein Shake,Single,1,0,1,0,1,0,0,0,0,0,1,0,0,1,1,0,1,0,0,0
7,P008,Protein Bar,Single,1,0,1,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0
137,P138,Meal Replacement,Single,0,0,0,1,1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0
101,P102,Healthy Snack,4-Pack,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1


In [31]:
from google.colab import files
df.to_csv('product_cleaned1.csv', index=False)
files.download('product_cleaned1.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>